## Description: 
## 
### This script uses a list of station pairs (generated by p_10_filter_pot_duplicated_stations_add_infos.ipynb) that have close distance to each other <=0.25km.
### This list of station pairs contains some stations for that we don't have water quality data available. These stations are excluded from list of station pairs.
### Subsequently: all station pairs that have distance of 0km/same coordinates are filtered
### Then all 'connected' stations, that have same coordinates are assigned a new common ID 
### finally, the new IDs are assigned to the corresponding old Station_ID's in water quality dataframe and observed data are merged by calculating mean, if more than one value is  available for same observation date
## 

### required input files:
###                       - list of all pairs of potential duplicated stations (distance <=0.25km) (generated by (p_10_filter_pot_duplicated_stations_add_infos)
###                       - original stations shapefile/parquet: stations_info_HYBAS_HYRIV.parquet 
###                       - filtered and merged water quality data: filtered_merged_raw_data.csv
### output: contains: 
###                   1.) a list of all new station IDs and corresponding old station ID's
###                   2.) merged and aggregated water quality data (aggregated by obervation date and new_station ID)
###                   3.) shapefile containing all original stations and added column with new_assigned stations ID'S
###                   4.) shapefile containing only unique new station ID#s (duplicated stations are removed)

In [1]:
# load required modules:

import pandas as pd
from glob import glob
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point
import warnings
from tqdm import trange
from time import time
import pickle
from scipy.spatial.distance import cdist
from scipy.spatial import distance_matrix
import pdb
from math import sin, cos, sqrt, atan2, radians
from scipy.spatial.distance import pdist, squareform
from sklearn.metrics.pairwise import pairwise_distances
import multiprocessing
from multiprocessing import Pool
n_cores = multiprocessing.cpu_count()
import os
from tqdm import tqdm
from time import time

## 1. read in list of pot_duplicated_stations produced by p_10_filter_pot_duplicated_stations_add_infos
## set threshold: distance in km¶


In [2]:
thrs = 0.25 #else:  '05'

In [3]:
all_pairs_dup = pd.read_csv(f"../../output_data/merge_0km_dist_stations/all_pairs_pot_dup{thrs}km_with_infos.csv")
all_pairs_dup


,Unnamed: 0,x,y,distance_km,x_station_id,x_data_sourc,x_country,x_area,x_lat,x_lon,...,y_station_id,y_data_sourc,y_country,y_area,y_lat,y_lon,y_NEAR_FID,y_NEAR_DIST,same_country,same_river_id
0,0,"('GRQA', '100002')","('GRQA', 'USGS-01017100')",0.1470,"('GRQA', '100002')",GLORICH,USA,4.994408e+09,46.847909,-68.002459,...,"('GRQA', 'USGS-01017100')",WQP,USA,1.943000e+03,46.849209,-68.002804,7396181,55.003173,True,True
1,1,"('GRQA', '100003')","('GRQA', 'USGS-01017500')",0.0360,"('GRQA', '100003')",GLORICH,USA,5.949347e+09,46.773619,-67.831939,...,"('GRQA', 'USGS-01017500')",WQP,USA,2.301000e+03,46.773372,-67.831633,7398300,4.542234,True,True
2,2,"('GRQA', '100004')","('GRQA', '110876')",0.0124,"('GRQA', '100004')",GLORICH,USA,3.782731e+09,45.168718,-67.298037,...,"('GRQA', '110876')",GLORICH,Canada,3.782731e+09,45.168609,-67.298070,7453269,18.885529,False,True
3,3,"('GRQA', '100004')","('GRQA', '110877')",0.0124,"('GRQA', '100004')",GLORICH,USA,3.782731e+09,45.168718,-67.298037,...,"('GRQA', '110877')",GLORICH,Canada,3.782731e+09,45.168609,-67.298070,7453269,18.885529,False,True
4,4,"('GRQA', '100004')","('GRQA', 'CAN00052')",0.1286,"('GRQA', '100004')",GLORICH,USA,3.782731e+09,45.168718,-67.298037,...,"('GRQA', 'CAN00052')",GEMSTAT,Canada,-9.999000e+03,45.169720,-67.297220,7453269,17.538434,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15060,15060,"('sweden', '2')","('sweden', '7')",0.0146,"('sweden', '2')",https://data.krycklan.se,Sweden,-9.999000e+03,64.251425,19.777804,...,"('sweden', '7')",https://data.krycklan.se,Sweden,-9.999000e+03,64.251301,19.777900,1616828,2235.286633,True,True
15061,15061,"('sweden', '58')","('sweden', '59')",0.2270,"('sweden', '58')",https://data.krycklan.se,Sweden,-9.999000e+03,64.179048,19.863411,...,"('sweden', '59')",https://data.krycklan.se,Sweden,-9.999000e+03,64.177068,19.864548,1618649,406.903617,True,True
15062,15062,"('sweden', '6')","('sweden', '7')",0.2327,"('sweden', '6')",https://data.krycklan.se,Sweden,-9.999000e+03,64.250974,19.773143,...,"('sweden', '7')",https://data.krycklan.se,Sweden,-9.999000e+03,64.251301,19.777900,1616828,2235.286633,True,True
15063,15063,"('sweden', '60')","('sweden', '66')",0.1613,"('sweden', '60')",https://data.krycklan.se,Sweden,-9.999000e+03,64.172828,19.866116,...,"('sweden', '66')",https://data.krycklan.se,Sweden,-9.999000e+03,64.172863,19.862788,1618649,639.563847,True,True


## Read in observed water quality data:  filtered_merged_raw_data.csv

In [4]:
#--------------- WATER QUALITY DATA------------ ########################
# read in merged raw data:
tic = time()
wq_data = pd.read_csv('../../output_data/merged_datasets/filtered_merged_raw_data.csv')

wq_data = wq_data.set_index(['dataset', 'site_id']).replace(-9999,np.nan).sort_index()

# ----id's to string----

wq_data = wq_data.reset_index()
wq_data['site_id'] = wq_data['site_id'].astype(str)
wq_data = wq_data.set_index(['dataset', 'site_id']).sort_index()


wq_data['Station_ID'] = wq_data.index
wq_data['Station_ID'] = wq_data['Station_ID'].astype(str)

wq_data['obs_date'] = pd.DatetimeIndex(wq_data['obs_date'].values)


print("Block time [sec]: ",f'{time()-tic}')


C:\Users\bartusch\AppData\Local\Temp\ipykernel_6740\1903438787.py:4: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  wq_data = pd.read_csv('../../output_data/merged_datasets/filtered_merged_raw_data.csv')


Block time [sec]:  13.846012592315674


## read in Station data containing also info about close HYRIV_IDs and assigned HYDROBASINS:

In [5]:
# read in station data with added info about HYBAS_L12 and closest two HydroRiverID'S
stations = gpd.read_parquet(
    '../../output_data/assign_stations_HydroBasins/stations_info_HYBAS_HYRIV.parquet')

# create also in stations df a column 'Station_ID' by index:

stations['Station_ID1'] = stations.index
stations = stations.reset_index()
stations['Station_ID1'] = stations['Station_ID1'].astype(str)




## Add info about station assigned HYBAS_L12_ID to wq_data as column:

In [6]:
# add HydroBasins ID to water qualtity data by merging on Station_ID and Station_ID1: 
wq_data = wq_data.reset_index()

wq_data_basins = wq_data.merge(stations[['HYBAS_ID', 'Station_ID1']], how = 'left', left_on = 'Station_ID', right_on = 'Station_ID1')
wq_data_basins = wq_data_basins.set_index(['Station_ID']).drop('Station_ID1', axis = 1)


#### The station dataset contains more station_IDs than the water quality data: 
#### The stations DataFrame is therefore filtered to include only site IDs with available data:

In [7]:
# filter data: in station data are more stations than available in water quality data:

# get all unique station_ids of selected pairs
pot_dup_station_ids = pd.concat([all_pairs_dup['x'], all_pairs_dup['y']]).unique().tolist()
pot_dup_station_ids_set = set(pot_dup_station_ids)



# compare list of all unique station_ids of selected pairs to stations in observed water quality data: 

# next lines gives station id's which are not in our wq data included, but in station data:
# add temporary column of station ID's 
wq_data_basins['station_ID'] = wq_data_basins.index
not_available = pot_dup_station_ids_set.difference(wq_data_basins['station_ID'].astype(str))
# remove column 'station_ID again:
wq_data_basins.drop('station_ID', axis = 1)


# now remove station pairs that have at least in one column a station_id that is not availbale in wq_data: use & operator
# delete all rows from all_pairs_dup in which column 'X' or column 'y' have one of the elements in not available --> condidtion is negation of both column x AND column y have an element of not available
all_pairs_dup_available = all_pairs_dup[~all_pairs_dup['x'].isin(not_available) & ~all_pairs_dup['y'].isin(not_available)]

print(len(not_available)) # using threshold 0.5km --> 609 stations were not available, this reduced number of pairs of pot_duplicated_stations from 24258 pairs to 23123
print(len(all_pairs_dup))
print(len(all_pairs_dup_available)) 

print(f'Dataframe all_pairs_dup contains {len(all_pairs_dup)} pairs of stations with distance <= threshold of {thrs}km with {len(pot_dup_station_ids)} different station_ids. But for {len(not_available)} of these station ids are no observed data in raw_data available. Therefore list of station pairs must be filtered.')

583
15065
14093
Dataframe all_pairs_dup contains 15065 pairs of stations with distance <= threshold of 0.25km with 15226 different station_ids. But for 583 of these station ids are no observed data in raw_data available. Therefore list of station pairs must be filtered.


In [8]:
wq_data_basins.head(10)

,dataset,site_id,index,obs_date,NO3N,NH4N,NO2N,TOC,DOC,TP,...,TOC_F,DOC_F,TP_F,DIP_F,NO2N_NO3N,DIN,Q,OPO4,HYBAS_ID,station_ID
Station_ID,,,,,,,,,,,,,,,,,,,,,
"('GRQA', '100001')",GRQA,100001,0.0,1979-10-31,NaN,0.020002,NaN,14.999997,NaN,0.020009,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7120992500,"('GRQA', '100001')"
"('GRQA', '100001')",GRQA,100001,1.0,1979-12-12,NaN,0.040004,NaN,NaN,NaN,0.010004,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7120992500,"('GRQA', '100001')"
"('GRQA', '100001')",GRQA,100001,2.0,1980-01-29,NaN,0.020002,NaN,26.999995,NaN,0.020009,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7120992500,"('GRQA', '100001')"
"('GRQA', '100001')",GRQA,100001,3.0,1980-02-26,NaN,0.040004,NaN,NaN,25.999996,0.020009,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7120992500,"('GRQA', '100001')"
"('GRQA', '100001')",GRQA,100001,4.0,1980-04-30,NaN,0.040004,NaN,6.999999,NaN,0.020009,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7120992500,"('GRQA', '100001')"
"('GRQA', '100001')",GRQA,100001,5.0,1980-05-20,NaN,0.030003,NaN,NaN,590.000030,0.010004,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7120992500,"('GRQA', '100001')"
"('GRQA', '100001')",GRQA,100001,6.0,1980-06-25,NaN,0.119998,NaN,4.999999,NaN,0.020009,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7120992500,"('GRQA', '100001')"
"('GRQA', '100001')",GRQA,100001,7.0,1980-09-03,NaN,0.010001,NaN,NaN,5.500005,0.030013,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7120992500,"('GRQA', '100001')"
"('GRQA', '100001')",GRQA,100001,8.0,1980-09-23,NaN,0.010001,NaN,9.999998,NaN,0.020009,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7120992500,"('GRQA', '100001')"


In [9]:
# this filtering of all_pairs_dup_available reduced also number of station_ids, which have at least one other station in range <= threshold of 0.5/0.25 km
nstations = len(pd.concat([all_pairs_dup_available['x'], all_pairs_dup_available['y']]).unique().tolist())

print(f'After filtering the stations that are actually present in observed data, {len(all_pairs_dup_available)} station pairs with {nstations} different station_ids remain.')
# thrs: 0.5km: After filtering the stations that are actually present in observed data, 23123 station pairs with 20572 different station_ids remain.

After filtering the stations that are actually present in observed data, 14093 station pairs with 14324 different station_ids remain.


## First investigate pairs of potential duplicated stations with distance 0km:

#### --> merge and set new Station_ID

In [10]:

# subsest pairs with distance 0.0km:
pairs_dist0 =all_pairs_dup_available[all_pairs_dup_available['distance_km']==0]
ids = pd.concat([pairs_dist0['x'], pairs_dist0['y']]).unique().tolist()
print(len(ids))
print(f'In stations shapefile are {len(pairs_dist0)} pairs of stations which have a distance of 0.0km. They origin from {len(ids)} different station ids.')


#########ONLY FOR TESTING NOT RELEVANT FOR DATA PROCESSING:
# check how many stations have exactly the same coordinates? 
filter = pairs_dist0[(pairs_dist0['x_lat']==pairs_dist0['y_lat']) & (pairs_dist0['x_lon']==pairs_dist0['y_lon'])]
#filtered_rows = df[(df['x_lat'] == df['y_lat']) & (df['y_lon'] == df['x_lon'])]
print(len(filter))
print(len(pairs_dist0))
ids_filter = pd.concat([filter['x'], filter['y']]).unique().tolist()
print(len(ids_filter)) # if this is 2945 it  is equal according to result of st_intersection() function in R package 'sf'
#test0[['x', 'y', 'x_lat', 'x_lon', 'y_lat', 'y_lon']]


filter['diff_lat']  =abs(filter['x_lat']-filter['y_lat'])
filter['diff_lon'] = abs(filter['x_lon']-filter['y_lon'])
filter[['x', 'y', 'x_lat', 'x_lon', 'y_lat', 'y_lon', 'diff_lat']]
filter['diff_lon'].max()


print(len(all_pairs_dup))
print(len(pd.concat([all_pairs_dup['x'], all_pairs_dup['y']]).unique().tolist()))

3227
In stations shapefile are 4179 pairs of stations which have a distance of 0.0km. They origin from 3227 different station ids.
3923
4179
2763
15065
15226


C:\Users\bartusch\AppData\Local\Temp\ipykernel_6740\1012424309.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filter['diff_lat']  =abs(filter['x_lat']-filter['y_lat'])
C:\Users\bartusch\AppData\Local\Temp\ipykernel_6740\1012424309.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filter['diff_lon'] = abs(filter['x_lon']-filter['y_lon'])


## Function to identify connected stations: 

#### all stations with same coordinates get a new ID

In [12]:
# This function add to all "connected" stations, which have same coordinates or distance of 0km a new common station ID:


df_data =  pd.DataFrame({'Station_X':pairs_dist0['x'] , 'Station_Y': pairs_dist0['y']})
# function to find connected station ID's
def find_connected_components(edges):
    graph = {}
    for edge in edges.itertuples(index=False):
        if edge[0] not in graph:
            graph[edge[0]] = set()
        graph[edge[0]].add(edge[1])
        if edge[1] not in graph:
            graph[edge[1]] = set()
        graph[edge[1]].add(edge[0])

    visited = set()
    components = []

    def dfs(node, component):
        visited.add(node)
        component.append(node)
        for neighbor in graph[node]:
            if neighbor not in visited:
                dfs(neighbor, component)

    for node in graph:
        if node not in visited:
            component = []
            dfs(node, component)
            components.append(component)

    return components

# Find connected components
connected_components = find_connected_components(df_data[['Station_X', 'Station_Y']])


# create new data frame:

df_new_ids = pd.DataFrame(columns=['Stations', 'New_ID'])

for idx, component in enumerate(connected_components):
    component_df = pd.DataFrame({'New_ID': f'ID_0km_{idx + 1}', 'Stations': component})
    df_new_ids = pd.concat([df_new_ids, component_df], ignore_index=True)



connected_components
total_length = sum(len(sublist) for sublist in connected_components)
print(total_length)
pd.set_option('display.max_rows', None)





3227


### Save list of stations_ID that have same coordinates as another one and corresponding new_IDS as csv:

In [14]:
df_new_ids.to_csv('../../output_data/merge_0km_dist_stations/list_station_IDs_same_coordinates_assigned_NEW_IDs.csv')

### Add new ID's to the list of potential duplicated stations and the water quality data:

In [15]:
## Add new Station IDs to list of pairs: all_pairs_dup


# Assuming you have DataFrame 1 named 'df1' with columns 'x' and 'y', and DataFrame 2 named 'df2' with columns 'station_id' and 'new_ID'
all_pairs_dup_new_id0 = all_pairs_dup_available.copy()
all_pairs_dup_new_id0 = all_pairs_dup_new_id0.merge(df_new_ids[['Stations', 'New_ID']], left_on='x', right_on='Stations', how='left').rename(columns={'New_ID': 'new_id0_x'}).drop('Stations', axis=1)
all_pairs_dup_new_id0 = all_pairs_dup_new_id0.merge(df_new_ids[['Stations', 'New_ID']], left_on='y', right_on='Stations', how='left').rename(columns={'New_ID': 'new_id0_y'}).drop('Stations', axis=1)

################# Add new ID's to water quality data as new column:
wq_data_new_ID0 = wq_data_basins.copy()

# convert column station_ID to string

wq_data_new_ID0['station_ID'] = wq_data_new_ID0['station_ID'].astype(str)

# reset index
wq_data_new_ID0 = wq_data_new_ID0.reset_index().drop('Station_ID', axis = 'columns')

# drop column index
wq_data_new_ID0 = wq_data_new_ID0.drop('index', axis = 'columns')

wq_data_new_ID0 = wq_data_new_ID0.merge(df_new_ids[['Stations', 'New_ID']], left_on = 'station_ID', right_on = 'Stations', how = 'left').rename(columns = {'New_ID':'new_id0'})






#### Look at one example:

In [16]:
wq_data_new_ID0[wq_data_new_ID0['new_id0']=='ID_0km_10']

,dataset,site_id,obs_date,NO3N,NH4N,NO2N,TOC,DOC,TP,DIP,...,TP_F,DIP_F,NO2N_NO3N,DIN,Q,OPO4,HYBAS_ID,station_ID,Stations,new_id0
127029,GRQA,103236,2007-06-05,NaN,0.010001,0.000994,NaN,5.299998,0.006288,0.003655,...,NaN,NaN,NaN,NaN,NaN,NaN,7120687820,"('GRQA', '103236')","('GRQA', '103236')",ID_0km_10
127030,GRQA,103237,2007-06-05,NaN,0.010001,0.001541,NaN,14.699999,0.035712,0.003562,...,NaN,NaN,NaN,NaN,NaN,NaN,7120687820,"('GRQA', '103237')","('GRQA', '103237')",ID_0km_10


#### Now aggregate water quality data by new IDs for stations with same coordinates: 
* this is done by calculating mean of numeric observed values:
* Subset wq_data_new_id and keep at first only rows which are duplicated in obs_date and new_id_all:

In [17]:
# add new id to water quality data: add new column with new IDs  depending on  origin stations id's --> if no new ID use origin id:

wq_data_new_ID0['new_id0_all'] = wq_data_new_ID0['new_id0'].fillna(wq_data_new_ID0['station_ID'])
#wq_data_new_ID0 = wq_data_new_ID0.drop(['Station_ID', 'Station_ID1'], axis = 'columns')


# Subset wq_data_new_ID0: keep only rows that are duplicated in new_id0_all and obs_Date
wq_data_new_ID0_2 =wq_data_new_ID0.copy()
rows_dup = wq_data_new_ID0_2.duplicated(subset=['new_id0_all', 'obs_date'], keep=False)


duplicated_new_ID0 = wq_data_new_ID0_2[rows_dup]


# convert site_id to string type: 
duplicated_new_ID0['site_id'] = duplicated_new_ID0['site_id'].astype(str)

# set non numeric columns --> to treat as non numeric during aggregation--> take first value
# keep first value by aggregation:
non_numeric_columns = ['new_id0','HYBAS_ID', 'year']

# keep all values by aggregation:
agg_columns = ['dataset', 'site_id', 'Stations']

# set the numeric columns --> to treat as numeric during aggregation --> mean
numeric_columns = ['NO3N', 'NH4N', 'NO2N', 'TOC', 'DOC', 'TP', 'DIP', 'NO3N_F', 'NH4N_F', 'NO2N_F', 'TOC_F', 'DOC_F', 'TP_F', 'DIP_F', 'NO2N_NO3N', 'DIN', 'Q','OPO4']

# Define the aggregation methods for each column
aggregation = {column: 'mean' for column in numeric_columns}
aggregation.update({column: 'first' for column in non_numeric_columns})
aggregation.update({column: 'unique' for column in agg_columns})

#  group data by new_id0_all and obs_date and aggregate duplicated rows by calculationg mean:
aggregated_data0km = duplicated_new_ID0.groupby(['new_id0_all', 'obs_date'], as_index = True).agg(aggregation).reset_index()



# rename columns to concat with other data: 
aggregated_data0km = aggregated_data0km.drop(['site_id', 'new_id0'], axis = 'columns')
aggregated_data0km = aggregated_data0km.rename(columns = {'new_id0_all':'Site_id', 'Stations': 'merged_origins'})

# convert site_id to string type:   
#aggregated_data['site_id'] = aggregated_data['site_id'].astype(str)

len(wq_data_new_ID0)
len(aggregated_data0km) # 100071
len(duplicated_new_ID0) #207495
aggregated_data0km[1:10]

C:\Users\bartusch\AppData\Local\Temp\ipykernel_6740\1109706196.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  duplicated_new_ID0['site_id'] = duplicated_new_ID0['site_id'].astype(str)


,Site_id,obs_date,NO3N,NH4N,NO2N,TOC,DOC,TP,DIP,NO3N_F,...,TP_F,DIP_F,NO2N_NO3N,DIN,Q,OPO4,HYBAS_ID,year,dataset,merged_origins
1,ID_0km_1000,1992-05-22,0.273,NaN,0.0020,NaN,NaN,0.075,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,7121026680,1992,[GRQA],"[('GRQA', 'USGS-06412600'), ('GRQA', 'USGS-064..."
2,ID_0km_1000,1992-06-23,0.249,NaN,0.0045,NaN,NaN,0.035,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,7121026680,1992,[GRQA],"[('GRQA', 'USGS-06412600'), ('GRQA', 'USGS-064..."
3,ID_0km_1001,1981-11-09,NaN,NaN,NaN,NaN,NaN,0.050,NaN,NaN,...,NaN,NaN,NaN,NaN,0.0,0.03,7120511400,1981,"[GRQA, USGS]","[('GRQA', 'USGS-06610000'), ('USGS', '06610000')]"
4,ID_0km_1001,1982-01-04,NaN,NaN,NaN,NaN,NaN,0.060,NaN,NaN,...,NaN,NaN,NaN,NaN,0.0,0.03,7120511400,1982,"[GRQA, USGS]","[('GRQA', 'USGS-06610000'), ('USGS', '06610000')]"
5,ID_0km_1001,1982-03-01,NaN,NaN,NaN,NaN,NaN,0.190,NaN,NaN,...,NaN,NaN,NaN,NaN,0.0,0.12,7120511400,1982,"[GRQA, USGS]","[('GRQA', 'USGS-06610000'), ('USGS', '06610000')]"
6,ID_0km_1001,1982-05-03,NaN,NaN,NaN,NaN,NaN,0.080,NaN,NaN,...,NaN,NaN,NaN,NaN,0.0,0.07,7120511400,1982,"[GRQA, USGS]","[('GRQA', 'USGS-06610000'), ('USGS', '06610000')]"
7,ID_0km_1001,1982-07-19,NaN,NaN,NaN,NaN,NaN,4.200,NaN,NaN,...,NaN,NaN,NaN,NaN,0.0,0.07,7120511400,1982,"[GRQA, USGS]","[('GRQA', 'USGS-06610000'), ('USGS', '06610000')]"
8,ID_0km_1001,1982-09-01,NaN,NaN,NaN,NaN,NaN,0.150,NaN,NaN,...,NaN,NaN,NaN,NaN,0.0,0.02,7120511400,1982,"[GRQA, USGS]","[('GRQA', 'USGS-06610000'), ('USGS', '06610000')]"
9,ID_0km_1001,1982-11-01,NaN,NaN,NaN,NaN,NaN,0.140,NaN,NaN,...,NaN,NaN,NaN,NaN,0.0,0.04,7120511400,1982,"[GRQA, USGS]","[('GRQA', 'USGS-06610000'), ('USGS', '06610000')]"


### Now subset wq_data_new_ID0 and keep only not duplicated rows:
### subsequently concat not_duplicated_rows and aggregated_data0km:



In [18]:
# delete duplicated rows from wq_data_sorted and add new rows of dataframe aggregated data:

# subset wq_data_new_ID0 and keep only not duplicated rows:
wq_data_new_ID0_copy = wq_data_new_ID0.copy()
not_duplicated_wq_data_ID0km = wq_data_new_ID0_copy[~rows_dup]
#not_duplicated_wq_data_ID0km = not_duplicated_wq_data.reset_index()
not_duplicated_wq_data_ID0km = not_duplicated_wq_data_ID0km.drop(['station_ID', 'site_id', 'new_id0'], axis = 'columns')
not_duplicated_wq_data_ID0km = not_duplicated_wq_data_ID0km.rename(columns = {'new_id0_all':'Site_id','Stations':'merged_origins'})

# set column order corresponding not duplicated_wq_data: 
col_order = not_duplicated_wq_data_ID0km.columns.to_list()
aggregated_data0km = aggregated_data0km.reindex(columns=col_order)


# append aggregated data to not_duplicated_wq_data:
#filtered_wq_data = not_duplicated_wq_data.append(aggregated_data, ignore_index=True)
all_wq_data_agg_0km_dist = pd.concat([not_duplicated_wq_data_ID0km, aggregated_data0km], ignore_index=True)
print(len(all_wq_data_agg_0km_dist))
print(len(wq_data_new_ID0))


3206674
3314094


In [19]:
all_wq_data_agg_0km_dist['Site_id'].nunique()


71380

### Now add the new ID's also to the stations in the shapefile:

In [20]:
stations_new_ids = stations.copy()
stations_new_ids['site_id'] = stations_new_ids['site_id'].astype(str)
stations_new_ids['dataset'] = stations_new_ids['dataset'].astype(str)
stations_new_ids = stations_new_ids.set_index(['dataset', 'site_id'])
stations_new_ids['origin_ID']=stations_new_ids.index
stations_new_ids['origin_ID'] = stations_new_ids['origin_ID'].astype(str)
stations_new_ids = stations_new_ids.reset_index()
# merge stations with df_new_ids
stations_new_IDs_0km = stations_new_ids.merge(df_new_ids[['Stations', 'New_ID']], left_on='origin_ID', right_on='Stations', how='left').rename(columns={'New_ID': 'new_id0km'}).drop('Stations', axis=1)

# add a column which combines new ids with origin id, if no new ID is available: 
stations_new_IDs_0km['ID_updated'] = stations_new_IDs_0km['new_id0km'].fillna(stations_new_IDs_0km['origin_ID'])




# merge additional information about origin merged stations and origin datasets to stations_new_IDs_0km:

# group by 'ID_updated' and aggregate dataset and origin_ID--> keep all unique values
grouped_df = stations_new_IDs_0km.groupby('ID_updated').agg({
    'dataset': lambda x: list(set(x)),
    'origin_ID': lambda x: list(set(x)),
    'HYBAS_ID': lambda x: list(set(x))
}).reset_index()

# rename der aggregated columns:
grouped_df = grouped_df.rename(columns={'dataset': 'merged_dataset', 'origin_ID': 'merged_ids', 'HYBAS_ID':'merged_HYBASID'})

# merge with stations_new_IDs_0km  based on column 'ID_updated':
stations_new_IDs_0km_2 = pd.merge(stations_new_IDs_0km, grouped_df, on='ID_updated', how='left')

stations_new_IDs_0km_2['Station_ID'] = stations_new_IDs_0km_2['ID_updated'].astype(str)
stations_new_IDs_0km_2 = stations_new_IDs_0km_2.drop(['dataset', 'site_id'], axis = 'columns')



# create a second df: containing only one version per new_station ID: filtering duplicated rows based on ID_updated: 
unique_newIDs_stations_0km = stations_new_IDs_0km_2[~stations_new_IDs_0km_2['ID_updated'].duplicated(keep='first')]





#stations_new_ids[1:10]
#df_new_ids[1:10]
stations_new_IDs_0km_2[stations_new_IDs_0km_2['new_id0km']=='ID_0km_1']
#unique_newIDs_stations_0km[unique_newIDs_stations_0km['new_id0km']=='ID_0km_1']

,data_sourc,country,area,lat,lon,geometry,FID,HYBAS_ID,1stNEAR_FID,1stNEAR_DIST_m,...,median_str_ord,n_river_seg_basin,Station_ID1,origin_ID,new_id0km,ID_updated,merged_dataset,merged_ids,merged_HYBASID,Station_ID
54113,GLORICH,USA,169997674.3,38.968664,-77.047796,POINT (-77.0478 38.96866),54113,7120567550,7626284,2.419449,...,1.0,13.0,"('GRQA', '100586')","('GRQA', '100586')",ID_0km_1,ID_0km_1,[GRQA],"[('GRQA', '100587'), ('GRQA', '100586')]",[7120567550],ID_0km_1
59655,GLORICH,USA,169997674.3,38.968664,-77.047796,POINT (-77.0478 38.96866),59655,7120567550,7626284,2.419449,...,1.0,13.0,"('GRQA', '100587')","('GRQA', '100587')",ID_0km_1,ID_0km_1,[GRQA],"[('GRQA', '100587'), ('GRQA', '100586')]",[7120567550],ID_0km_1


### Save results: 
* 1.) aggregated water quality data with infos about new station_IDs 
* 2.) a new shapefile with all new_IDS stations ID's (containing all original stations with added new ID)
* 3.) a filtered version containing only unique new ids + not duplicated original station ids


In [21]:
# Save data: 

folder_out = '../../output_data/merge_0km_dist_stations/'



# 1.) aggregated water quality data with infos about new station_IDs¶

all_wq_data_agg_0km_dist = all_wq_data_agg_0km_dist.drop('year', axis = 1)
all_wq_data_agg_0km_dist.to_csv(f'{folder_out}wq_data_aggregated_0km_dist_new_ID.csv')

# 2.) a new shapefile with all stations containing new_IDS and original stations ID's (containing all original stations with added new ID)
stations_new_IDs_0km_2['merged_ids'] = stations_new_IDs_0km_2['merged_ids'].astype(str)
stations_new_IDs_0km_2['merged_dataset'] = stations_new_IDs_0km_2['merged_dataset'].astype(str)
stations_new_IDs_0km_2['merged_HYBASID'] = stations_new_IDs_0km_2['merged_HYBASID'].astype(str)

stations_new_IDs_0km_2.to_file(f'{folder_out}stations_aggregated_IDs_0km_dist.shp')
stations_new_IDs_0km_2.to_parquet(f'{folder_out}stations_aggregated_IDs_0km_dist.parquet')
stations_new_IDs_0km_2.to_csv(f'{folder_out}stations_aggregated_IDs_0km_dist.csv')
# 3.) save fitered version of stations: containing only unique version of all stations:
unique_newIDs_stations_0km['merged_ids'] = unique_newIDs_stations_0km['merged_ids'].astype(str)
unique_newIDs_stations_0km['merged_dataset'] = unique_newIDs_stations_0km['merged_dataset'].astype(str)
unique_newIDs_stations_0km['merged_HYBASID'] = unique_newIDs_stations_0km['merged_HYBASID'].astype(str)



unique_newIDs_stations_0km.to_file(f'{folder_out}unique_stations_aggregated_IDs_0km_dist.shp')
unique_newIDs_stations_0km.to_parquet(f'{folder_out}unique_stations_aggregated_IDs_0km_dist.parquet')
unique_newIDs_stations_0km.to_csv(f'{folder_out}unique_stations_aggregated_IDs_0km_dist.csv')

C:\Users\bartusch\AppData\Local\Temp\ipykernel_6740\1302671562.py:17: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  stations_new_IDs_0km_2.to_file(f'{folder_out}stations_aggregated_IDs_0km_dist.shp')
c:\Users\bartusch\cnp_env_local\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: '1stNEAR_FID' to '1stNEAR_FI'
  ogr_write(
c:\Users\bartusch\cnp_env_local\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: '1stNEAR_DIST_m' to '1stNEAR_DI'
  ogr_write(
c:\Users\bartusch\cnp_env_local\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: '1stHYRIV_ID' to '1stHYRIV_I'
  ogr_write(
c:\Users\bartusch\cnp_env_local\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: '1stORD_STRA' to '1stORD_STR'
  ogr_write(
c:\Users\bartusch\cnp_env_local\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normali

## Now check data availability: 

In [21]:

# convert df to long format: 
all_wq_data_agg_0km_dist_2 = all_wq_data_agg_0km_dist.copy()
all_wq_data_agg_0km_dist_2['Station_ID'] = all_wq_data_agg_0km_dist_2['Site_id']
all_wq_data_agg_0km_dist_2['year'] = all_wq_data_agg_0km_dist_2['obs_date'].dt.year
all_wq_data_agg_0km_dist_2 = all_wq_data_agg_0km_dist_2.drop(columns = ['NO3N_F', 'NO2N_F', 'NH4N_F', 'DOC_F', 'TOC_F', 'DIP_F', 'TP_F', 'Q', 'HYBAS_ID', 'merged_origins'], axis = 1)

df_long = pd.melt(all_wq_data_agg_0km_dist_2, id_vars=['obs_date', 'year', 'Station_ID', 'dataset'], var_name='fraction', value_name='value').dropna(subset=['value'])
#PER FRACTION:
    
    
# n observations , start year and end year per fraction:
stats_per_fraction = df_long.groupby('fraction')['year'].agg(['count', 'min', 'max'])
    
    
# n unique site ids per fraction:
n_stations_per_fraction = df_long.groupby('fraction')['Station_ID'].nunique()
    
    
    
# per fraction statistics: mean/median, range number of observations per station: 
mean_count_per_fraction_site = df_long.groupby(['fraction', 'Station_ID']).count().groupby(level=[0])['obs_date'].agg(['mean', 'median', 'max', 'min'])
mean_count_per_fraction_site
    
    

#mean and median time series length in years; median/mean start year; maximum time series length years per station;
    
stats_years_station_frac = df_long.groupby(['Station_ID', 'fraction'])['year'].nunique().groupby(level=[1]).agg(['mean', 'median', 'max', 'min']).rename(columns = {'mean':'mean_station_years',
                                                                                                                                                                 'median':'median_station_years',
                                                                                                                                                                 'max':'max_station_years',
                                                                                                                                                                 'min':'min_station_years'})
    
# median number of samples per station and year and fraction
median_obs_fraction_station_year = df_long.groupby(['Station_ID', 'fraction','year'])['value'].count().groupby(level = [1]).median()
merged_df1 = stats_years_station_frac.merge(median_obs_fraction_station_year, left_index=True, right_index=True).rename(columns = {'value':'median_n_obs_per_year'})
merged_df2 = merged_df1.merge(stats_per_fraction,  left_index=True, right_index=True).rename(columns = {'count':'n_obs','min':'start_year','max':'end_year'})
merged_df3 = merged_df2.merge(mean_count_per_fraction_site, left_index=True, right_index=True).rename(columns = {'mean':'mean_obs_station',
                                                                                                                     'median':'median_obs_station',
                                                                                                                     'max':'max_obs_station',
                                                                                                                     'min':'min_obs_station'})
stats_fractions = merged_df3.merge(n_stations_per_fraction, left_index=True, right_index=True).rename(columns = {'Station_ID':'n_sites'})

stats_fractions = stats_fractions.reset_index()
   


In [22]:
stats_fractions.T

,0,1,2,3,4,5,6,7,8,9,10
fraction,DIN,DIP,DOC,NH4N,NO2N,NO2N_NO3N,NO3N,OPO4,Site_id,TOC,TP
mean_station_years,2.0,6.605801,4.397346,5.607157,4.194265,9.24,4.777843,22.719178,5.144368,4.369618,4.842691
median_station_years,2.0,3.0,2.0,2.0,2.0,10.0,2.0,22.0,2.0,2.0,2.0
max_station_years,2,52,43,52,55,15,77,41,82,50,56
min_station_years,2,1,1,1,1,3,1,10,1,1,1
median_n_obs_per_year,12.5,10.0,6.0,8.0,5.0,25.0,5.0,11.0,6.0,6.0,6.0
n_obs,100,1253616,844907,1101436,908187,5080,1929322,138942,3206674,632303,1996675
start_year,2008,1942,1958,1942,1900,2004,1900,1966,1900,1905,1900
end_year,2009,2022,2022,2022,2020,2021,2022,2013,2022,2020,2022
mean_obs_station,25.0,73.156863,34.706991,61.488081,26.654937,203.2,36.724507,475.828767,44.923984,29.401237,40.648921
